# Grupo 4 — MLP Regressor para temperatura em Jena

Implementação das tarefas de Dmitri: T03, T07, T13 e T18. As células de integração com T06/T08/T14 apenas conectam os entregáveis aos demais responsáveis.

### Informações do Grupo

##### Grupo: 4

Base: `jena_climate_2009_2016.csv`

Série meteorológica de Jena, agregada de 10 minutos para frequência horária. O alvo é a temperatura do ar `T (degC)`.

- Granularidade: horária
- Random State (seed): 42
- Treino-teste: 80% inicial para treino e 20% final para teste

## Importações e Configurações Padrão 

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted



import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "validacao_bases.py").is_file():
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")
sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
DATA_FILENAME = "jena_climate_2009_2016.csv"
DATA_PATH = PROJECT_ROOT / "grupo4" / DATA_FILENAME
DATE_COLUMN = "Date Time"
TARGET_COLUMN = "T (degC)"
FREQUENCY = "1h"
TRAIN_RATIO = 0.80
SEED = 42
HORIZON = 1
SEASONAL_PERIOD = 24
FFILL_LIMIT_HOURS = 6
BASE_EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd (deg)"]
EXOG_COLUMNS = ["p (mbar)", "rh (%)", "wv (m/s)", "wd_sin", "wd_cos"]
OUT_ROOT = PROJECT_ROOT / "outputs"
print(f"Base: {DATA_PATH}")


## Configuração do MLP Regressor

In [ ]:
# Configuração da MLP para previsão causal da próxima hora.
TARGET_LAGS = [1, 2, 3, 6, 12, 24, 48, 168]
EXOG_LAGS = {coluna: [1, 24] for coluna in EXOG_COLUMNS}
ROLLING_WINDOWS = [6, 24, 168]
SCALER = 'standard'
PARAM_GRID = [
    {'hidden_layer_sizes': [(64, 64)], 'activation': ['relu'], 'solver': ['adam'], 'alpha': [1e-4], 'max_iter': [500]},
    {'hidden_layer_sizes': [(64, 64)], 'activation': ['relu'], 'solver': ['adam'], 'alpha': [1e-3], 'max_iter': [500]},
    {'hidden_layer_sizes': [(32,)], 'activation': ['relu'], 'solver': ['adam'], 'alpha': [1e-4], 'max_iter': [300]},
    {'hidden_layer_sizes': [(64, 32)], 'activation': ['relu'], 'solver': ['adam'], 'alpha': [1e-4], 'max_iter': [500]},
    {'hidden_layer_sizes': [(64, 32)], 'activation': ['tanh'], 'solver': ['adam'], 'alpha': [1e-3], 'max_iter': [500]},
    {'hidden_layer_sizes': [(32,)], 'activation': ['tanh'], 'solver': ['adam'], 'alpha': [1e-3], 'max_iter': [300]},
    {'hidden_layer_sizes': [(64, 64)], 'activation': ['relu'], 'solver': ['sgd'], 'alpha': [1e-4], 'max_iter': [500]},
    {'hidden_layer_sizes': [(64, 32)], 'activation': ['tanh'], 'solver': ['sgd'], 'alpha': [1e-3], 'max_iter': [500]},
]


# 1. Preparação da base

## T01 — Sanitização e congelamento da base

Congelamento e validação sanitária da base: verificação de nulos, duplicatas, datas, ordenação e frequência regular.

In [ ]:
# T01 — limpeza e agregação horária padronizadas no grupo 4.
df_bruto = pd.read_csv(DATA_PATH)
df_bruto[DATE_COLUMN] = pd.to_datetime(
    df_bruto[DATE_COLUMN], format="%d.%m.%Y %H:%M:%S", errors="coerce"
)
n_datas_invalidas = int(df_bruto[DATE_COLUMN].isna().sum())
n_duplicadas = int(df_bruto[DATE_COLUMN].duplicated().sum())
df_bruto = (
    df_bruto.dropna(subset=[DATE_COLUMN])
    .sort_values(DATE_COLUMN)
    .drop_duplicates(subset=[DATE_COLUMN], keep="first")
    .set_index(DATE_COLUMN)
)

# -9999 é ausência, portanto deve ser removido antes da média horária.
df_numerico = df_bruto.select_dtypes(include="number").replace(-9999, np.nan)
df = df_numerico.resample(FREQUENCY).mean()

# O alvo nunca é imputado. Somente exógenas recebem preenchimento causal.
alvo_ausente = df[TARGET_COLUMN].isna().copy()
for coluna in BASE_EXOG_COLUMNS:
    df[coluna] = df[coluna].ffill(limit=FFILL_LIMIT_HOURS)
df.loc[alvo_ausente, TARGET_COLUMN] = np.nan

# Direção é circular: 0 e 360 graus representam a mesma direção.
df["wd_sin"] = np.sin(np.deg2rad(df["wd (deg)"]))
df["wd_cos"] = np.cos(np.deg2rad(df["wd (deg)"]))

if not df.index.is_monotonic_increasing or not df.index.is_unique:
    raise ValueError("O índice horário deve ser crescente e único.")
if (df["wv (m/s)"].dropna() < 0).any():
    raise ValueError("Há velocidade do vento negativa depois da limpeza.")

# O corte é definido na série horária completa, antes dos lags dos modelos.
posicao_corte = int(len(df) * TRAIN_RATIO)
FIM_TREINO = df.index[posicao_corte - 1]
INICIO_TESTE = df.index[posicao_corte]
diagnostico_base = pd.DataFrame([{
    "linhas_horarias": len(df),
    "datas_invalidas": n_datas_invalidas,
    "duplicatas_removidas": n_duplicadas,
    "horas_sem_alvo": int(df[TARGET_COLUMN].isna().sum()),
    "frequencia": FREQUENCY,
    "ffill_limite_horas": FFILL_LIMIT_HOURS,
    "fim_treino": FIM_TREINO,
    "inicio_teste": INICIO_TESTE,
}])
display(diagnostico_base)
print(f"Período: {df.index.min()} a {df.index.max()}")
print("O alvo não foi imputado; exógenas usam ffill causal limitado.")


## T02 — Dicionário de dados e disponibilidade temporal

Criação do dicionário de dados e análise de disponibilidade temporal das variáveis exógenas para prevenir data leakage.

In [ ]:
# T02 — integração: disponibilidade usada para impedir data leakage
dicionario = pd.DataFrame([
    {
        'variavel': coluna,
        'papel': 'alvo' if coluna == TARGET_COLUMN else 'exogena',
        'tipo': str(df[coluna].dtype),
        'uso_no_modelo': 'somente valores defasados; calendário é conhecido antecipadamente',
    }
    for coluna in [TARGET_COLUMN] + EXOG_COLUMNS
]).set_index('variavel')
display(dicionario)

## T03 — Análise exploratória visual

Análise visual do alvo e das variáveis exógenas, diagnóstico de outliers e registro das decisões de limpeza.

In [ ]:
# T03 — Dmitri: diagnóstico visual e sinais de outlier (sem remoção automática)
def diagnosticar_exploracao(dados, colunas, fator_iqr=1.5):
    faltantes = [coluna for coluna in colunas if coluna not in dados.columns]
    if faltantes:
        raise KeyError(f'Colunas ausentes na base: {faltantes}')
    linhas = []
    for coluna in colunas:
        serie = pd.to_numeric(dados[coluna], errors='coerce')
        validos = serie.dropna()
        q1, q3 = validos.quantile([0.25, 0.75])
        iqr = q3 - q1
        inferior, superior = q1 - fator_iqr * iqr, q3 + fator_iqr * iqr
        outliers = ((validos < inferior) | (validos > superior)).sum()
        linhas.append({
            'variavel': coluna, 'n': int(validos.size),
            'ausentes': int(serie.isna().sum()), 'media': float(validos.mean()),
            'desvio_padrao': float(validos.std()), 'minimo': float(validos.min()),
            'q1': float(q1), 'mediana': float(validos.median()), 'q3': float(q3),
            'maximo': float(validos.max()), 'limite_iqr_inferior': float(inferior),
            'limite_iqr_superior': float(superior), 'sinais_outlier_iqr': int(outliers),
            'percentual_outlier_iqr': float(outliers / max(1, validos.size)),
        })
    return pd.DataFrame(linhas).set_index('variavel')

colunas_t03 = [TARGET_COLUMN] + EXOG_COLUMNS
resumo_t03 = diagnosticar_exploracao(df, colunas_t03)
display(resumo_t03.round(3))

fig, axes = plt.subplots(len(colunas_t03), 1, figsize=(14, 12), sharex=True)
for ax, coluna in zip(axes, colunas_t03):
    df[coluna].resample('1D').mean().plot(ax=ax, linewidth=0.8, label=coluna)
    ax.set_ylabel(coluna)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.25)
axes[-1].set_xlabel('Data')
fig.suptitle('T03 — Jena Climate: alvo e exógenas (médias diárias)', y=1.01)
plt.tight_layout()
out_t03 = OUT_ROOT / 't03' / 'grupo4'
out_t03.mkdir(parents=True, exist_ok=True)
resumo_t03.to_csv(out_t03 / 'resumo_mlp.csv')
fig.savefig(out_t03 / 'series_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

print('Decisão: preservar valores extremos fisicamente plausíveis; não aplicar clipping global.')
print('Na modelagem, o alvo não é imputado; somente exógenas usam forward-fill causal de até 6 horas.')

# 2. Sazonalidade

## T04 — Decomposição STL

Decomposição STL e interpretação gráfica da tendência, incluindo períodos de alta, queda e estabilidade.

In [ ]:
# T04
# Importe e execute aqui a função reutilizável desta tarefa.
# Mantenha somente resultados essenciais para a análise.

## T05 — Força da sazonalidade e resíduos

Cálculo da força da sazonalidade e análise do componente residual.

In [ ]:
# T05
# Importe e execute aqui a função reutilizável desta tarefa.
# Mantenha somente resultados essenciais para a análise.

# 3. Features e validação

## T06 — Pipeline de features

Construção do pipeline de features com lags do alvo e das exógenas, janelas móveis, variáveis de calendário e encoding cíclico.

In [ ]:
def construir_features_temporais(
    dados, alvo, *, lags_alvo, lags_exogenas, janelas_moveis
):
    """Cria features causais; o índice da linha representa a hora prevista."""
    features = pd.DataFrame(index=dados.index)
    for lag in sorted(set(lags_alvo)):
        if lag < HORIZON:
            raise ValueError(f"Lag do alvo deve ser >= {HORIZON}: {lag}")
        features[f"target_lag_{lag}"] = dados[alvo].shift(lag)
    for coluna, lags in lags_exogenas.items():
        for lag in sorted(set(lags)):
            if lag < HORIZON:
                raise ValueError(f"Lag da exógena deve ser >= {HORIZON}: {lag}")
            features[f"{coluna}_lag_{lag}"] = dados[coluna].shift(lag)
    passado = dados[alvo].shift(HORIZON)
    for janela in sorted(set(janelas_moveis)):
        features[f"target_roll_mean_{janela}"] = passado.rolling(janela).mean()
        features[f"target_roll_std_{janela}"] = passado.rolling(janela).std()
    features["hora_sin"] = np.sin(2 * np.pi * dados.index.hour / 24)
    features["hora_cos"] = np.cos(2 * np.pi * dados.index.hour / 24)
    features["dia_semana_sin"] = np.sin(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_semana_cos"] = np.cos(2 * np.pi * dados.index.dayofweek / 7)
    features["dia_ano_sin"] = np.sin(2 * np.pi * dados.index.dayofyear / 365.25)
    features["dia_ano_cos"] = np.cos(2 * np.pi * dados.index.dayofyear / 365.25)
    return pd.concat([dados[alvo].rename("target"), features], axis=1).dropna()


model_data = construir_features_temporais(
    df, TARGET_COLUMN,
    lags_alvo=TARGET_LAGS,
    lags_exogenas=EXOG_LAGS,
    janelas_moveis=ROLLING_WINDOWS,
)

train = model_data.loc[model_data.index < INICIO_TESTE]
test = model_data.loc[model_data.index >= INICIO_TESTE]
if train.empty or test.empty or train.index.max() >= test.index.min():
    raise ValueError("Divisão temporal inválida.")
y_train, y_test = train["target"], test["target"]
X_train, X_test = train.drop(columns="target"), test.drop(columns="target")

print(f"Features: {X_train.shape[1]} | treino: {len(train):,} | teste final: {len(test):,}")
print(f"Corte comum: {FIM_TREINO} -> {INICIO_TESTE}")


## T07 — Escalonamento por janela

Padronização e escalonamento com MinMaxScaler ou StandardScaler ajustado estritamente no treino de cada janela.

In [ ]:
# T07 — Dmitri: prova explícita de escala ajustada só na janela de treino
@dataclass(frozen=True)
class ResultadoEscalonamento:
    X_treino: pd.DataFrame
    X_avaliacao: pd.DataFrame
    escalonador_X: Any

def validar_X(X, nome):
    if not isinstance(X, pd.DataFrame) or X.empty:
        raise ValueError(f'{nome} deve ser um DataFrame não vazio')
    nao_numericas = X.select_dtypes(exclude='number').columns.tolist()
    if nao_numericas:
        raise TypeError(f'{nome} contém colunas não numéricas: {nao_numericas}')
    if np.isinf(X.to_numpy(dtype=float)).any():
        raise ValueError(f'{nome} contém valores infinitos')
    if isinstance(X.index, pd.DatetimeIndex) and not X.index.is_monotonic_increasing:
        raise ValueError(f'{nome} deve estar em ordem temporal crescente')

def criar_escalonador(tipo):
    if tipo == 'standard':
        return StandardScaler()
    if tipo == 'minmax':
        return MinMaxScaler()
    raise ValueError("tipo deve ser 'standard' ou 'minmax'")

def escalonar_janela(X_treino, X_avaliacao, tipo='standard'):
    validar_X(X_treino, 'X_treino')
    validar_X(X_avaliacao, 'X_avaliacao')
    if list(X_treino.columns) != list(X_avaliacao.columns):
        raise ValueError('Treino e avaliação devem possuir as mesmas features')
    scaler = criar_escalonador(tipo)
    X_treino_sc = pd.DataFrame(
        scaler.fit_transform(X_treino), index=X_treino.index, columns=X_treino.columns
    )
    X_avaliacao_sc = pd.DataFrame(
        scaler.transform(X_avaliacao), index=X_avaliacao.index, columns=X_avaliacao.columns
    )
    return ResultadoEscalonamento(X_treino_sc, X_avaliacao_sc, scaler)

n_validacao = max(24 * 30, int(len(X_train) * 0.15))
X_ajuste, X_validacao = X_train.iloc[:-n_validacao], X_train.iloc[-n_validacao:]
y_ajuste, y_validacao = y_train.iloc[:-n_validacao], y_train.iloc[-n_validacao:]
janela_escalada = escalonar_janela(X_ajuste, X_validacao, tipo=SCALER)
np.testing.assert_allclose(
    janela_escalada.escalonador_X.mean_, X_ajuste.mean().to_numpy(), rtol=1e-10
)
diagnostico_escala = pd.DataFrame({
    'media_treino_escalado': janela_escalada.X_treino.mean(),
    'media_validacao_escalada': janela_escalada.X_avaliacao.mean(),
})
display(diagnostico_escala.head(10).round(4))
out_t07 = OUT_ROOT / 't07' / 'grupo4'
out_t07.mkdir(parents=True, exist_ok=True)
diagnostico_escala.to_csv(out_t07 / 'diagnostico_escalonamento.csv')
print('Validação concluída: fit no ajuste; validação recebeu apenas transform().')

## T08 — Validação walk-forward

Implementação do protocolo walk-forward com janelas, origens de teste fixas e horizonte de previsão unificado.

In [ ]:
# T08 — teste final fixo e validação temporal somente dentro do treino.
print(f'Fim do treino: {X_train.index.max()}')
print(f'Início do teste final: {X_test.index.min()}')
print('Horizonte: 1 hora. Busca: TimeSeriesSplit com 3 divisões no treino.')


# 4. Modelagem MLP

## T13 — Arquitetura e otimização da MLP

Busca de camadas, neurônios, ativação, solver, alpha e max_iter sem acessar o conjunto de teste final. A arquitetura `(64, 64)` do material é a referência, comparada com redes mais simples. No `MLPRegressor`, a saída já é linear. O early stopping usa os 15% finais de cada janela como validação temporal, `patience=20` e restauração do melhor estado.

In [ ]:
# T13 — Dmitri: arquitetura e otimização usando apenas o conjunto de treino
class MLPRegressorEarlyStoppingTemporal(RegressorMixin, BaseEstimator):
    def __init__(
        self, hidden_layer_sizes=(64, 64), activation='relu', solver='adam',
        alpha=0.0001, batch_size=32, learning_rate_init=0.001, max_iter=500,
        validation_fraction=0.15, n_iter_no_change=20, tol=0.0001,
        random_state=42, shuffle=True
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.activation = activation
        self.solver = solver
        self.alpha = alpha
        self.batch_size = batch_size
        self.learning_rate_init = learning_rate_init
        self.max_iter = max_iter
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.tol = tol
        self.random_state = random_state
        self.shuffle = shuffle

    def fit(self, X, y):
        Xv, yv = check_X_y(X, y, y_numeric=True)
        if self.solver not in {'adam', 'sgd'}:
            raise ValueError("solver deve ser 'adam' ou 'sgd'")
        corte = int(len(Xv) * (1 - self.validation_fraction))
        if corte < 2 or len(Xv) - corte < 1:
            raise ValueError('Janela pequena demais para validação temporal')
        X_fit, X_val, y_fit, y_val = Xv[:corte], Xv[corte:], yv[:corte], yv[corte:]
        modelo = MLPRegressor(
            hidden_layer_sizes=self.hidden_layer_sizes, activation=self.activation,
            solver=self.solver, alpha=self.alpha, batch_size=self.batch_size,
            learning_rate_init=self.learning_rate_init, max_iter=1,
            early_stopping=False, random_state=self.random_state, shuffle=self.shuffle,
        )
        self.loss_curve_, self.validation_loss_curve_ = [], []
        melhor_loss, melhor_modelo, sem_melhora = np.inf, None, 0
        for epoca in range(1, self.max_iter + 1):
            modelo.partial_fit(X_fit, y_fit)
            mae_fit = mean_absolute_error(y_fit, modelo.predict(X_fit))
            mae_val = mean_absolute_error(y_val, modelo.predict(X_val))
            self.loss_curve_.append(float(mae_fit))
            self.validation_loss_curve_.append(float(mae_val))
            if mae_val < melhor_loss - self.tol:
                melhor_loss, melhor_modelo = float(mae_val), deepcopy(modelo)
                self.best_epoch_, sem_melhora = epoca, 0
            else:
                sem_melhora += 1
                if sem_melhora >= self.n_iter_no_change:
                    break
        self.model_ = melhor_modelo
        self.n_iter_ = len(self.loss_curve_)
        self.best_validation_loss_ = melhor_loss
        self.n_features_in_ = Xv.shape[1]
        return self

    def predict(self, X):
        check_is_fitted(self, 'model_')
        return self.model_.predict(check_array(X))

@dataclass(frozen=True)
class ResultadoOtimizacao:
    modelo: GridSearchCV
    melhores_parametros: dict
    resultados: pd.DataFrame

def criar_pipeline_mlp(tipo_escalonador='standard', seed=42):
    return Pipeline([
        ('imputador', SimpleImputer(strategy='median')),
        ('escalonador', criar_escalonador(tipo_escalonador)),
        ('mlp', MLPRegressorEarlyStoppingTemporal(
            random_state=seed, validation_fraction=0.15,
            n_iter_no_change=20, batch_size=32
        )),
    ])

def otimizar_mlp_temporal(
    X_treino, y_treino, grade_parametros, *, tipo_escalonador='standard',
    n_splits=3, gap=0, max_train_size=None, seed=42, n_jobs=None, verbose=0
):
    validar_X(X_treino, 'X_treino')
    prefixo = 'mlp__'
    def prefixar(grade):
        return {
            chave if chave.startswith(prefixo) else f'{prefixo}{chave}': valores
            for chave, valores in grade.items()
        }
    grade = [prefixar(item) for item in grade_parametros]
    cv = TimeSeriesSplit(n_splits=n_splits, gap=gap, max_train_size=max_train_size)
    busca = GridSearchCV(
        criar_pipeline_mlp(tipo_escalonador, seed), grade,
        scoring='neg_mean_absolute_error', cv=cv, refit=True, n_jobs=n_jobs,
        verbose=verbose, return_train_score=True, error_score='raise',
    )
    busca.fit(X_treino, y_treino)
    tabela = pd.DataFrame(busca.cv_results_)
    tabela['MAE_validacao'] = -tabela['mean_test_score']
    tabela['desvio_MAE_validacao'] = tabela['std_test_score']
    tabela = tabela[[
        'rank_test_score', 'MAE_validacao', 'desvio_MAE_validacao',
        'mean_fit_time', 'params'
    ]].sort_values('rank_test_score').reset_index(drop=True)
    melhores = {k.removeprefix(prefixo): v for k, v in busca.best_params_.items()}
    return ResultadoOtimizacao(busca, melhores, tabela)

resultado_busca = otimizar_mlp_temporal(
    X_train,
    y_train,
    PARAM_GRID,
    tipo_escalonador=SCALER,
    n_splits=3,
    max_train_size=24 * 365 * 2,
    seed=SEED,
    n_jobs=-1,
    verbose=1,
)
display(resultado_busca.resultados.head(8))
print('Melhores hiperparâmetros:', resultado_busca.melhores_parametros)

out_t13 = OUT_ROOT / 't13' / 'grupo4'
out_t13.mkdir(parents=True, exist_ok=True)
resultado_busca.resultados.to_csv(out_t13 / 'grid_mlp.csv', index=False)
(out_t13 / 'melhores_parametros.json').write_text(
    json.dumps(resultado_busca.melhores_parametros, indent=2), encoding='utf-8'
)

## T14 — MLP walk-forward

Treinamento e validação walk-forward da MLP Regressor, garantindo escalonamento ajustado em cada janela de treino. Seguindo o material, são apresentados MAE, MSE, RMSE, R², curvas treino × validação e real × previsto. MAPE não é interpretável para temperaturas zero/negativas; por isso também é reportado sMAPE.

In [ ]:
# T14 — integração para Julia: refit no treino completo e avaliação fora da amostra
def avaliar_regressao(modelo, X, y, conjunto='teste'):
    validar_X(X, 'X')
    previsto = pd.Series(modelo.predict(X), index=y.index, name='previsao')
    residuo = (y - previsto).rename('residuo')
    mse = mean_squared_error(y, previsto)
    denominador = np.abs(y.to_numpy()) + np.abs(previsto.to_numpy())
    smape = np.mean(np.divide(
        2 * np.abs(y.to_numpy() - previsto.to_numpy()), denominador,
        out=np.zeros_like(denominador, dtype=float), where=denominador != 0
    )) * 100
    mape = (
        np.mean(np.abs((y.to_numpy() - previsto.to_numpy()) / y.to_numpy())) * 100
        if np.all(y.to_numpy() > 0) else np.nan
    )
    metricas = pd.DataFrame([{
        'conjunto': conjunto, 'n': len(y), 'MAE': mean_absolute_error(y, previsto),
        'MSE': mse, 'RMSE': mse ** 0.5, 'MAPE_percent': mape,
        'sMAPE_percent': smape, 'R2': r2_score(y, previsto),
        'vies_medio': residuo.mean(),
    }])
    return metricas, previsto, residuo

def extrair_historico_treino(modelo):
    mlp = modelo.named_steps['mlp']
    check_is_fitted(mlp, 'model_')
    historico = pd.DataFrame({
        'epoca': np.arange(1, mlp.n_iter_ + 1),
        'MAE_treino': mlp.loss_curve_,
        'MAE_validacao': mlp.validation_loss_curve_,
    })
    resumo = {
        'epocas_executadas': mlp.n_iter_, 'melhor_epoca': mlp.best_epoch_,
        'melhor_MAE_validacao': mlp.best_validation_loss_,
        'patience': mlp.n_iter_no_change, 'restaurou_melhor_estado': True,
    }
    return historico, resumo

modelo_mlp = resultado_busca.modelo.best_estimator_
metricas_treino, previsao_treino, residuos_treino = avaliar_regressao(
    modelo_mlp, X_train, y_train, conjunto='treino'
)
metricas_mlp, previsao_mlp, residuos_mlp = avaliar_regressao(
    modelo_mlp, X_test, y_test, conjunto='teste_final'
)
metricas_comparacao = pd.concat([metricas_treino, metricas_mlp], ignore_index=True)
display(metricas_comparacao.round(4))
print('MAPE fica NaN porque a temperatura contém valores zero/negativos; use sMAPE.')
previsoes_mlp = pd.concat([y_test.rename('observado'), previsao_mlp, residuos_mlp], axis=1)
display(previsoes_mlp.head())

historico_mlp, resumo_early_stopping = extrair_historico_treino(modelo_mlp)
display(pd.DataFrame([resumo_early_stopping]))
out_t14 = OUT_ROOT / 't14' / 'grupo4'
out_t14.mkdir(parents=True, exist_ok=True)
metricas_comparacao.to_csv(out_t14 / 'metricas_mlp.csv', index=False)
historico_mlp.to_csv(out_t14 / 'historico_loss_mlp.csv', index=False)
previsoes_mlp.to_csv(out_t14 / 'previsoes_residuos_mlp.csv')
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(historico_mlp['epoca'], historico_mlp['MAE_treino'], label='treino')
ax.plot(historico_mlp['epoca'], historico_mlp['MAE_validacao'], label='validação temporal')
ax.axvline(resumo_early_stopping['melhor_epoca'], color='black', linestyle='--', label='melhor época')
ax.set(title='Evolução da loss (MAE)', xlabel='Época', ylabel='MAE (°C)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
fig.savefig(out_t14 / 'curva_loss_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

## T15 — Síntese técnica da MLP

Registro da intuição, hipóteses, limitações, curva de aprendizado e comparação entre bases para a MLP Regressor.

In [ ]:
# T15 — subsídios para Sofia: evidências produzidas por T13/T14
print('Arquitetura selecionada:', resultado_busca.melhores_parametros['hidden_layer_sizes'])
print('Saída: linear, adequada à previsão contínua de temperatura.')
print('Treino interno: erro quadrático; seleção da época: MAE de validação temporal.')
print('Early stopping: patience=20 e restauração do melhor estado.')
print('A MLP modela relações não lineares entre lags, exógenas e ciclos de calendário.')
print('Limitações: custo de ajuste, sensibilidade à escala e extrapolação fraca fora do domínio observado.')
display(metricas_comparacao.round(4))

# 5. Avaliação

## T16 — Comparação por MAE

Consolidação do MAE fora da amostra por base e modelo, incluindo número de vitórias e posição média.

In [ ]:
# T16
# Importe e execute aqui a função reutilizável desta tarefa.
# Mantenha somente resultados essenciais para a análise.

## T17 — Diagnóstico dos resíduos

Análise gráfica e estatística dos resíduos com série temporal dos erros, ACF, teste de Ljung-Box e tabela de p-valores.

In [ ]:
# T17 — integração para Felipe: diagnósticos gráficos recomendados no material da MLP
amostra_treino = previsao_treino.sample(n=min(5000, len(previsao_treino)), random_state=SEED).index
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_train.loc[amostra_treino], previsao_treino.loc[amostra_treino], alpha=0.25, s=10, label='treino')
axes[0].scatter(y_test, previsao_mlp, alpha=0.35, s=10, label='teste')
mn = min(y_train.min(), y_test.min(), previsao_treino.min(), previsao_mlp.min())
mx = max(y_train.max(), y_test.max(), previsao_treino.max(), previsao_mlp.max())
axes[0].plot([mn, mx], [mn, mx], '--', color='black', label='linha y=x')
axes[0].set(title='Real vs. previsto', xlabel='Temperatura real (°C)', ylabel='Temperatura prevista (°C)')
axes[0].legend()

axes[1].scatter(previsao_treino.loc[amostra_treino], residuos_treino.loc[amostra_treino], alpha=0.25, s=10, label='treino')
axes[1].scatter(previsao_mlp, residuos_mlp, alpha=0.35, s=10, label='teste')
axes[1].axhline(0, linestyle='--', color='black')
axes[1].set(title='Resíduos vs. previsto', xlabel='Temperatura prevista (°C)', ylabel='Resíduo: real - previsto (°C)')
axes[1].legend()

axes[2].hist(residuos_mlp, bins=30, density=True, alpha=0.8)
axes[2].axvline(0, linestyle='--', color='black')
axes[2].set(title='Histograma dos resíduos de teste', xlabel='Resíduo (°C)', ylabel='Densidade')
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
out_t17 = OUT_ROOT / 't17' / 'grupo4'
out_t17.mkdir(parents=True, exist_ok=True)
fig.savefig(out_t17 / 'diagnosticos_graficos_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

## T18 — Importância das features

Cálculo e interpretação da importância das features: permutation importance para modelos de tabela e coeficientes para o SARIMAX.

In [ ]:
# T18 — Dmitri: permutation importance exclusivamente no teste final
def calcular_importancia_permutacao(
    modelo, X_avaliacao, y_avaliacao, n_repeticoes=10, seed=42, n_jobs=None
):
    validar_X(X_avaliacao, 'X_avaliacao')
    resultado = permutation_importance(
        modelo, X_avaliacao, y_avaliacao, scoring='neg_mean_absolute_error',
        n_repeats=n_repeticoes, random_state=seed, n_jobs=n_jobs,
    )
    return pd.DataFrame({
        'feature': X_avaliacao.columns,
        'aumento_MAE_medio': resultado.importances_mean,
        'desvio_aumento_MAE': resultado.importances_std,
    }).sort_values('aumento_MAE_medio', ascending=False).reset_index(drop=True)

n_amostra_importancia = min(3000, len(X_test))
X_importancia = X_test.sample(n=n_amostra_importancia, random_state=SEED).sort_index()
y_importancia = y_test.loc[X_importancia.index]
importancias_mlp = calcular_importancia_permutacao(
    modelo_mlp,
    X_importancia,
    y_importancia,
    n_repeticoes=10,
    seed=SEED,
    n_jobs=-1,
)
display(importancias_mlp.head(15).round(4))

top = importancias_mlp.head(15).sort_values('aumento_MAE_medio')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top['feature'], top['aumento_MAE_medio'], xerr=top['desvio_aumento_MAE'])
ax.set_title('T18 — Importância por permutação da MLP (teste final)')
ax.set_xlabel('Aumento do MAE após permutação (°C)')
out_t18 = OUT_ROOT / 't18' / 'grupo4'
out_t18.mkdir(parents=True, exist_ok=True)
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
fig.savefig(out_t18 / 'importancia_permutacao_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

importancias_mlp.to_csv(out_t18 / 'importancia_permutacao_mlp.csv', index=False)